In [ ]:
# ============================================================
#   SISTEMA DE MONITORAMENTO DO NÍVEL DO RIO NEGRO
#   Projeto: Curso de IA - CETAM | Módulo Python
# ============================================================

# --- FAIXAS DE REFERÊNCIA (em metros) ---
NIVEL_CHEIA      = 27.50   # Acima disso → CHEIA
NIVEL_SECA       = 14.50   # Abaixo disso → SECA
ALERTA_CHEIA     = 30.02   # Recorde histórico de cheia (2021/2022)
ALERTA_SECA      = 12.11   # Recorde histórico de seca (10/2024)

MESES = [
    "Janeiro", "Fevereiro", "Março",    "Abril",
    "Maio",    "Junho",     "Julho",    "Agosto",
    "Setembro","Outubro",   "Novembro", "Dezembro"
]


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def classificar_nivel(nivel: float) -> str:
    """Classifica o nível do rio em CHEIA, NORMAL ou SECA."""
    if nivel > NIVEL_CHEIA:   # Se nivel for MAIOR (27.50m)
        return "CHEIA"
    elif nivel < NIVEL_SECA:   # Senaõ se nivel MENOR (14.50m)
        return "SECA"
    else:
        return "NORMAL"   # Senão está normal o nível


def verificar_alerta(nivel: float) -> str | None:
    """Retorna mensagem de alerta se ultrapassar limites históricos."""
    if nivel > ALERTA_CHEIA:   # Se nivel MAIOR (30.02m)
        return (f"⚠️  ALERTA CRÍTICO! Nível {nivel:.2f}m SUPERA o recorde "
                f"histórico de CHEIA ({ALERTA_CHEIA}m - 2021/2022)!")
    elif nivel < ALERTA_SECA:   # Se nivel MENOR (12.11m)
        return (f"⚠️  ALERTA CRÍTICO! Nível {nivel:.2f}m está ABAIXO do recorde "
                f"histórico de SECA ({ALERTA_SECA}m)!")
    return None


def ler_nivel(prompt: str) -> float:
    """Lê e valida um nível do rio informado pelo usuário."""
    while True:
        try:
            valor = float(input(prompt))
            if valor < 0 or valor > 60:
                print("  ❌  Valor fora do intervalo realista (0 a 60 m). Tente novamente.")
                continue
            return valor
        except ValueError:
            print("  ❌  Entrada inválida. Digite um número (ex: 25.40).")


def coletar_dados_usuario(mes: str, num_dias: int) -> list[float]:
    """Coleta leituras diárias digitadas pelo usuário."""
    print(f"\n  Informe os níveis diários para {mes} ({num_dias} dias):")
    leituras = []
    for dia in range(1, num_dias + 1):
        nivel = ler_nivel(f"    Dia {dia:02d}: ")
        leituras.append(nivel)
    return leituras


def usar_dados_simulados(mes: str) -> list[float]:
    """Retorna dados fictícios pré-definidos para demonstração."""
    import random
    random.seed(42)
    # Simula oscilação realista baseada no comportamento do Rio Negro
    base = {
        "Janeiro": 25.0, "Fevereiro": 27.5, "Março": 29.5, "Abril": 30.2,
        "Maio": 29.8, "Junho": 28.0, "Julho": 25.5, "Agosto": 22.0,
        "Setembro": 18.5, "Outubro": 17.0, "Novembro": 18.5, "Dezembro": 22.0
    }
    nivel_base = base.get(mes, 24.0)
    leituras = []
    nivel_atual = nivel_base
    for _ in range(30):
        variacao = random.uniform(-0.6, 0.6)
        nivel_atual = round(nivel_atual + variacao, 2)
        nivel_atual = max(10.0, min(35.0, nivel_atual))
        leituras.append(nivel_atual)
    return leituras


# ============================================================
# ANÁLISE MENSAL
# ============================================================

def analisar_mes(mes: str, leituras: list[float]) -> dict:
    """Processa todas as leituras de um mês e retorna o relatório."""
    resultados_dias = []
    alertas = []

    for dia, nivel in enumerate(leituras, start=1):
        classificacao = classificar_nivel(nivel)
        alerta = verificar_alerta(nivel)
        resultados_dias.append({
            "dia": dia,
            "nivel": nivel,
            "classificacao": classificacao,
        })
        if alerta:
            alertas.append(f"    Dia {dia:02d}: {alerta}")

    # Média mensal
    media = sum(leituras) / len(leituras)

    # Dia de maior variação (diferença entre dias consecutivos)
    maior_variacao = 0.0
    dia_maior_variacao = 1
    for i in range(1, len(leituras)):
        variacao = abs(leituras[i] - leituras[i - 1])
        if variacao > maior_variacao:
            maior_variacao = variacao
            dia_maior_variacao = i + 1  # dia atual (base 1)

    # Contagem por classificação
    contagem = {"CHEIA": 0, "NORMAL": 0, "SECA": 0}
    for r in resultados_dias:
        contagem[r["classificacao"]] += 1

    return {
        "mes": mes,
        "leituras": leituras,
        "resultados_dias": resultados_dias,
        "media": media,
        "maior_variacao": maior_variacao,
        "dia_maior_variacao": dia_maior_variacao,
        "contagem": contagem,
        "alertas": alertas,
    }


# ============================================================
# EXIBIÇÃO DE RESULTADOS
# ============================================================

ICONE = {"CHEIA": "🌊", "NORMAL": "✅", "SECA": "🏜️ "}

def exibir_relatorio(rel: dict) -> None:
    """Imprime o relatório completo do mês."""
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  RELATÓRIO — {rel['mes'].upper()}")
    print(sep)

    # Leituras diárias
    print("\n  📅 LEITURAS DIÁRIAS:")
    for r in rel["resultados_dias"]:
        icone = ICONE[r["classificacao"]]
        print(f"    Dia {r['dia']:02d}: {r['nivel']:6.2f} m  → {icone}  {r['classificacao']}")

    # Resumo
    print(f"\n  📊 RESUMO DO MÊS:")
    print(f"    • Média mensal      : {rel['media']:.2f} m  ({classificar_nivel(rel['media'])})")
    print(f"    • Dias em CHEIA     : {rel['contagem']['CHEIA']}")
    print(f"    • Dias NORMAS      : {rel['contagem']['NORMAL']}")
    print(f"    • Dias em SECA      : {rel['contagem']['SECA']}")
    print(f"    • Maior variação    : {rel['maior_variacao']:.2f} m  (Dia {rel['dia_maior_variacao']:02d})")

    # Alertas históricos
    if rel["alertas"]:
        print(f"\n  🚨 ALERTAS HISTÓRICOS:")
        for a in rel["alertas"]:
            print(a)
    else:
        print(f"\n  ✅ Nenhum nível ultrapassou os limites históricos.")

    print(sep)


def exibir_resumo_anual(relatorios: list[dict]) -> None:
    """Exibe tabela-resumo de todos os meses analisados."""
    if not relatorios:
        return
    print("\n" + "=" * 60)
    print("  RESUMO ANUAL — RIO NEGRO")
    print("=" * 60)
    print(f"  {'Mês':<12} {'Média (m)':>10} {'Classificação':<10} {'Alertas':>7}")
    print("  " + "-" * 46)
    for r in relatorios:
        classe = classificar_nivel(r["media"])
        icone = ICONE[classe]
        qtd_alertas = len(r["alertas"])
        print(f"  {r['mes']:<12} {r['media']:>10.2f} {icone} {classe:<9} {qtd_alertas:>6}")
    print("=" * 60)


# ============================================================
# MENU PRINCIPAL
# ============================================================

def menu_principal() -> None:
    print("\n" + "=" * 60)
    print("   🌊  MONITORAMENTO DO NÍVEL DO RIO NEGRO  🌊")
    print("   Projeto CETAM — Curso de IA | Módulo Python")
    print("=" * 60)
    print("\n  MENU PRINCIPAL")
    print("  1 - Analisar um mês (dados simulados)")
    print("  2 - Analisar um mês (inserir dados manualmente)")
    print("  3 - Gerar relatório anual completo (simulado)")
    print("  0 - Sair")
    print()


def escolher_mes() -> tuple[str, int]:
    """Permite o usuário escolher o mês."""
    print("\n  Escolha o mês:")
    for i, m in enumerate(MESES, 1):
        print(f"    {i:2d}. {m}")
    while True:
        try:
            escolha = int(input("\n  Digite o número do mês: "))
            if 1 <= escolha <= 12:
                mes = MESES[escolha - 1]
                # Dias por mês (simplificado, sem ano bissexto)
                dias_por_mes = [31,28,31,30,31,30,31,31,30,31,30,31]
                return mes, dias_por_mes[escolha - 1]
            else:
                print("  ❌  Número inválido. Escolha entre 1 e 12.")
        except ValueError:
            print("  ❌  Entrada inválida. Digite um número.")


def main() -> None:
    relatorios_acumulados = []

    while True:
        menu_principal()
        try:
            opcao = input("  Opção: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\n  Programa encerrado. Até logo! 👋")
            break

        if opcao == "0":
            print("\n  Programa encerrado. Até logo! 👋\n")
            break

        elif opcao == "1":
            mes, num_dias = escolher_mes()
            leituras = usar_dados_simulados(mes)
            relatorio = analisar_mes(mes, leituras)
            exibir_relatorio(relatorio)
            relatorios_acumulados.append(relatorio)

        elif opcao == "2":
            mes, num_dias = escolher_mes()
            leituras = coletar_dados_usuario(mes, num_dias)
            relatorio = analisar_mes(mes, leituras)
            exibir_relatorio(relatorio)
            relatorios_acumulados.append(relatorio)

        elif opcao == "3":
            print("\n  ⏳ Gerando relatório anual com dados simulados...")
            relatorios_anuais = []
            for mes in MESES:
                leituras = usar_dados_simulados(mes)
                rel = analisar_mes(mes, leituras)
                relatorios_anuais.append(rel)
                exibir_relatorio(rel)
            exibir_resumo_anual(relatorios_anuais)

        else:
            print("\n  ❌  Opção inválida. Tente novamente.")

        input("\n  [Pressione ENTER para continuar...]")


# ============================================================
# PONTO DE ENTRADA
# ============================================================

if __name__ == "__main__":
    main()


   🌊  MONITORAMENTO DO NÍVEL DO RIO NEGRO  🌊
   Projeto CETAM — Curso de IA | Módulo Python

  MENU PRINCIPAL
  1 - Analisar um mês (dados simulados)
  2 - Analisar um mês (inserir dados manualmente)
  3 - Gerar relatório anual completo (simulado)
  0 - Sair


  Escolha o mês:
     1. Janeiro
     2. Fevereiro
     3. Março
     4. Abril
     5. Maio
     6. Junho
     7. Julho
     8. Agosto
     9. Setembro
    10. Outubro
    11. Novembro
    12. Dezembro
